# 02 · Bicyclist Casualty, Demographic, Condition & Co-Involved Vehicle Trends

This notebook analyzes national patterns across `2010–2023` (NCDB Online) and `2010–2021` (linked public microdata):

- **Long-Term Decline & 2023 Rebound**: Bicycle involvements fell `27.0%` from `6,712` in `2010` to `4,902` in `2023`, but surged `+21.1%` in `2023` over `2022` (`4,047 -> 4,902`, standing `+5.9%` above `2019`'s `4,628`).
- **Seasonal & Intersection Concentration**: `May–September` accounts for `70.2%` of all bicycle involvements (`53,489 / 76,153`), and public-roadway intersections account for `61.1%` (`46,547 / 76,153`).
- **Co-Involved Vehicle Lethality Asymmetry**: Light-duty vehicles (`V_TYPE='01'`) are present in `89.82%` of unique bicycle collisions (`61,698 / 68,687`) with a `0.708%` fatality rate, whereas heavy unit trucks (`7.623%`, `10.76x`) and road tractors (`13.355%`, `18.86x`) account for `20.36%` of fatal bicycle collisions despite only `2.05%` collision share.

In [1]:
import csv
from pathlib import Path
from bicycle_collisions.corridor_exposure_link import compute_national_co_involved_lethality

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src/bicycle_collisions").exists():
            return candidate
    raise RuntimeError("Could not locate projects/bicycles root")

ROOT = find_project_root()
inv = {int(r["year"]): r for r in csv.DictReader((ROOT / "reports/tables/annual_bicycle_involvements.csv").open())}
cas = {int(r["year"]): r for r in csv.DictReader((ROOT / "reports/tables/annual_bicyclist_casualties.csv").open())}

v2010, v2019, v2022, v2023 = (int(inv[y]["vehicles"]) for y in (2010, 2019, 2022, 2023))
pct_2010_2023 = 100.0 * (v2023 - v2010) / v2010
pct_2022_2023 = 100.0 * (v2023 - v2022) / v2022
pct_2019_2023 = 100.0 * (v2023 - v2019) / v2019

print(f"Bicycle involvements: 2010={v2010:,} | 2019={v2019:,} | 2022={v2022:,} | 2023={v2023:,}")
print(f"Changes: 2010->2023: {pct_2010_2023:+.1f}% | 2022->2023: {pct_2022_2023:+.1f}% | 2019->2023: {pct_2019_2023:+.1f}%")
print(f"2023 Bicyclist casualties (Road User query): {int(cas[2023]['fatalities'])} deaths, {int(cas[2023]['injured']):,} nonfatal injuries")
print(f"2010-2023 Cumulative bicyclist deaths: {sum(int(r['fatalities']) for r in cas.values()):,} | Nonfatal injuries: {sum(int(r['injured']) for r in cas.values()):,}")

Bicycle involvements: 2010=6,712 | 2019=4,628 | 2022=4,047 | 2023=4,902
Changes: 2010->2023: -27.0% | 2022->2023: +21.1% | 2019->2023: +5.9%
2023 Bicyclist casualties (Road User query): 46 deaths, 4,742 nonfatal injuries
2010-2023 Cumulative bicyclist deaths: 712 | Nonfatal injuries: 75,610


## 2. Seasonal, Temporal, Roadway Configuration & Lighting Profiles (2010–2023)

We aggregate the `2010–2023` condition tables to verify the concentration of bicycle involvements during warm months (`May–September`), at public-roadway intersections, during afternoon peak hours (`15:00–17:59`), and across lighting conditions.

In [2]:
def sum_by_category(table_name: str, metric: str = "vehicles") -> dict[str, int]:
    rows = list(csv.DictReader((ROOT / "reports/tables" / table_name).open()))
    out = {}
    for r in rows:
        cat = r["category"]
        out[cat] = out.get(cat, 0) + int(r[metric])
    return out

months_v = sum_by_category("month_2010_2023.csv", "vehicles")
total_v = sum(months_v.values())
warm_months = ["May", "June", "July", "August", "September"]
warm_v = sum(months_v[m] for m in warm_months)

road_cfg_v = sum_by_category("road_configuration_2010_2023.csv", "vehicles")
intersection_cat = "At an intersection of at least two public roadways"
non_int_cat = "Non-intersection"

hour_v = sum_by_category("hour_2010_2023.csv", "vehicles")
light_v = sum_by_category("light_2010_2023.csv", "vehicles")
light_f = sum_by_category("light_2010_2023.csv", "fatalities")

print(f"Total 2010-2023 bicycle involvements: {total_v:,}")
print(f"May-September involvements: {warm_v:,} ({100.0 * warm_v / total_v:.1f}%)")
print(f"Public-road intersections:  {road_cfg_v[intersection_cat]:,} ({100.0 * road_cfg_v[intersection_cat] / total_v:.1f}%)")
print(f"Non-intersection midblock:  {road_cfg_v[non_int_cat]:,} ({100.0 * road_cfg_v[non_int_cat] / total_v:.1f}%)")
print(f"Peak 3-hour window (15:00 to 17:59): {hour_v['15:00 to 17:59']:,} ({100.0 * hour_v['15:00 to 17:59'] / total_v:.1f}%)")
print(
    f"Daylight involvements: {light_v['Daylight']:,} ({100.0 * light_v['Daylight'] / total_v:.1f}% of involvements, "
    f"{100.0 * light_f['Daylight'] / sum(light_f.values()):.1f}% of fatalities)"
)
print(
    f"Darkness involvements: {light_v['Darkness']:,} ({100.0 * light_v['Darkness'] / total_v:.1f}% of involvements, "
    f"{100.0 * light_f['Darkness'] / sum(light_f.values()):.1f}% of fatalities)"
)

Total 2010-2023 bicycle involvements: 77,153
May-September involvements: 54,173 (70.2%)
Public-road intersections:  47,175 (61.1%)
Non-intersection midblock:  18,129 (23.5%)
Peak 3-hour window (15:00 to 17:59): 22,038 (28.6%)
Daylight involvements: 61,292 (79.4% of involvements, 67.9% of fatalities)
Darkness involvements: 11,613 (15.1% of involvements, 26.5% of fatalities)


## 3. Demographics, All-Road-User Harm & Co-Involved Vehicle Lethality (2010–2021)

We examine age and gender distributions (`age_complete_2010_2023.csv`, `gender_2010_2023.csv`), all road users harmed in bicycle-involved collisions (`people_harmed_2010_2021.csv`), and the lethality multiplier across co-involved motor vehicle types (`compute_national_co_involved_lethality`).

In [3]:
gender_f = sum_by_category("gender_2010_2023.csv", "fatalities")
gender_i = sum_by_category("gender_2010_2023.csv", "injured")
print(
    f"Gender split (2010-2023): Male={gender_f['Male']} deaths ({100.0 * gender_f['Male'] / sum(gender_f.values()):.1f}%), "
    f"{gender_i['Male']:,} injured ({100.0 * gender_i['Male'] / sum(gender_i.values()):.1f}%) | "
    f"Female={gender_f['Female']} deaths, {gender_i['Female']:,} injured"
)

harm_rows = list(csv.DictReader((ROOT / "reports/tables/people_harmed_2010_2021.csv").open()))
print("\nAll-Road-User Harm in Bicycle Collisions (2010-2021 Linked Microdata):")
for r in harm_rows[:4]:
    print(
        f"  {r['road_user']:25s}: identified={int(r['identified_people']):6,d} | "
        f"resolved_deaths={int(r['deaths_resolved']):3d} | resolved_injuries={int(r['injuries_resolved']):6,d}"
    )

co_veh = compute_national_co_involved_lethality(ROOT)
print("\nCo-Involved Vehicle Lethality in Unique Bicycle Collisions (2010-2021, N=68,687):")
for r in co_veh:
    print(
        f"  {r['vehicle_group']:42s}: collisions={r['collisions']:6,d} ({r['share_of_all_bicycle_collisions_pct']:5.2f}%) | "
        f"fatal={r['fatal_collisions']:3d} ({r['share_of_fatal_bicycle_collisions_pct']:5.2f}% of fatal) | "
        f"fatality_rate={r['fatality_rate_pct']:6.3f}% ({r['lethality_multiplier_vs_light_duty']:5.2f}x LDV)"
    )

Gender split (2010-2023): Male=567 deaths (79.6%), 49,989 injured (66.1%) | Female=135 deaths, 18,464 injured

All-Road-User Harm in Bicycle Collisions (2010-2021 Linked Microdata):
  Bicyclist                : identified=72,610 | resolved_deaths=610 | resolved_injuries=66,849
  Pedestrian               : identified= 1,480 | resolved_deaths=  7 | resolved_injuries=   859
  Motor vehicle driver     : identified=65,732 | resolved_deaths=  3 | resolved_injuries= 1,403
  Motorcyclist             : identified=   544 | resolved_deaths=  2 | resolved_injuries=   279

Co-Involved Vehicle Lethality in Unique Bicycle Collisions (2010-2021, N=68,687):
  Light-duty vehicle                        : collisions=61,698 (89.82%) | fatal=437 (71.17% of fatal) | fatality_rate= 0.708% ( 1.00x LDV)
  Cargo or other light truck/van            : collisions= 2,415 ( 3.52%) | fatal= 34 ( 5.54% of fatal) | fatality_rate= 1.408% ( 1.99x LDV)
  Heavy unit truck                          : collisions= 1,102 ( 1.60%